# HDB Rental Data Preprocessing

This notebook converts the raw competition files into model-ready tables using the findings from `eda.ipynb`.

Key decisions:

- preserve raw CSV files and row order;
- retain duplicates and target extremes by default (they may represent repeated transactions and genuine market values);
- normalize inconsistent text labels such as `4 room` versus `4-room`;
- treat time explicitly and use chronological validation later;
- merge block and geospatial data with deterministic keys;
- lag COE and stock features by one month so only information available before the rental month is used;
- never create or export `RENT_PER_SQM`, because it directly contains the target;
- fit data-dependent rules (for example, rare-category grouping) on training data only.

Outputs are written to `data/processed/` and can be reproduced by running all cells from top to bottom.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.neighbors import BallTree

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

RANDOM_STATE = 42
EARTH_RADIUS_KM = 6_371.0088
RARE_MODEL_MIN_COUNT = 100
SAVE_OUTPUTS = True


def locate_project_root() -> Path:
    # Find the repository root whether the notebook starts in root or notebooks/.
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "CS5228/project/cs5228-group6",
        cwd.parent / "cs5228-group6",
    ]
    for candidate in candidates:
        if (candidate / "data/train.csv").exists() and (candidate / "data/test.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate a project root containing data/train.csv and data/test.csv")


PROJECT_ROOT = locate_project_root()
DATA_DIR = PROJECT_ROOT / "data"
AUX_DIR = DATA_DIR / "auxiliary"
OUTPUT_DIR = DATA_DIR / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Auxiliary data: {AUX_DIR}")

## 1. Load raw data and enforce the data contract

The checks below fail early if columns change or if IDs cannot preserve the Kaggle test-row order.

In [ ]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")

expected_features = [
    "RENT_APPROVAL_DATE", "TOWN", "BLOCK", "STREET", "FLAT_TYPE",
    "FLAT_MODEL", "FLOOR_AREA_SQM", "FURNISHED", "LEASE_COMMENCE_DATE", "FEE",
]
target = "MONTHLY_RENT"

assert train_raw.columns.tolist() == expected_features + [target]
assert test_raw.columns.tolist() == expected_features
assert train_raw[target].notna().all()

train_raw.insert(0, "ROW_ID", np.arange(len(train_raw), dtype=np.int64))
test_raw.insert(0, "ROW_ID", np.arange(len(test_raw), dtype=np.int64))

audit = pd.DataFrame(
    {
        "rows": [len(train_raw), len(test_raw)],
        "columns": [train_raw.shape[1], test_raw.shape[1]],
        "missing_cells": [int(train_raw.isna().sum().sum()), int(test_raw.isna().sum().sum())],
        "exact_duplicate_rows": [
            int(train_raw.drop(columns="ROW_ID").duplicated().sum()),
            int(test_raw.drop(columns="ROW_ID").duplicated().sum()),
        ],
    },
    index=["train", "test"],
)
display(audit)

# EDA found repeated transactions and only ~0.5% target IQR outliers. Keep both.
print("Policy: no rows removed during preprocessing.")

## 2. Reusable cleaning helpers

Dates support both formats observed across earlier file versions (`2021-01` and `Jan-21`). Text normalization is deliberately conservative: casing, whitespace, punctuation variants, and Unicode are standardized without inventing values.

In [ ]:
def parse_rent_month(values: pd.Series) -> pd.Series:
    text = values.astype("string").str.strip()
    parsed = pd.Series(pd.NaT, index=values.index, dtype="datetime64[ns]")
    for fmt in ("%Y-%m", "%b-%y"):
        missing = parsed.isna()
        if not missing.any():
            break
        parsed.loc[missing] = pd.to_datetime(text.loc[missing], format=fmt, errors="coerce")
    if parsed.isna().any():
        bad = text.loc[parsed.isna()].drop_duplicates().tolist()[:10]
        raise ValueError(f"Unrecognized rental month values: {bad}")
    return parsed.dt.to_period("M").dt.to_timestamp()


def normalize_text(value):
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKC", str(value)).strip().lower()
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_block(value):
    value = normalize_text(value)
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", "", value)


def normalize_flat_type(value):
    value = normalize_text(value)
    if pd.isna(value):
        return pd.NA
    value = value.replace("-", " ")
    value = re.sub(r"\s+", " ", value).strip()
    aliases = {
        "1 room": "1-room", "2 room": "2-room", "3 room": "3-room",
        "4 room": "4-room", "5 room": "5-room", "executive": "executive",
    }
    return aliases.get(value, value.replace(" ", "-"))


def clean_number(values: pd.Series) -> pd.Series:
    return pd.to_numeric(
        values.astype("string").str.replace(r"[$,]", "", regex=True).str.strip(),
        errors="coerce",
    )


def month_index(values: pd.Series, base=pd.Timestamp("2021-01-01")) -> pd.Series:
    return (values.dt.year - base.year) * 12 + (values.dt.month - base.month)


FLAT_TYPE_ORDER = {
    "1-room": 1, "2-room": 2, "3-room": 3,
    "4-room": 4, "5-room": 5, "executive": 6,
}

## 3. Core-table preprocessing

`FURNISHED` and `FEE` are dropped because EDA showed they are constant. The original approval-date string is replaced with a parsed month plus explicit trend and seasonal features. One negative lease-age record is retained with a flag while its invalid age value is set to missing.

In [ ]:
def prepare_core(df: pd.DataFrame, rare_models: set[str]) -> pd.DataFrame:
    out = df.copy()
    out["RENT_DATE"] = parse_rent_month(out["RENT_APPROVAL_DATE"])
    out["RENT_YEAR"] = out["RENT_DATE"].dt.year.astype("int16")
    out["RENT_MONTH"] = out["RENT_DATE"].dt.month.astype("int8")
    out["RENT_QUARTER"] = out["RENT_DATE"].dt.quarter.astype("int8")
    out["MONTH_INDEX"] = month_index(out["RENT_DATE"]).astype("int16")
    out["MONTH_SIN"] = np.sin(2 * np.pi * out["RENT_MONTH"] / 12)
    out["MONTH_COS"] = np.cos(2 * np.pi * out["RENT_MONTH"] / 12)

    for column in ["TOWN", "STREET", "FLAT_MODEL"]:
        out[column] = out[column].map(normalize_text).astype("string")
    out["BLOCK"] = out["BLOCK"].map(normalize_block).astype("string")
    out["FLAT_TYPE"] = out["FLAT_TYPE"].map(normalize_flat_type).astype("string")

    out["PROPERTY_KEY"] = (
        out["TOWN"].fillna("") + "|" + out["BLOCK"].fillna("") + "|" + out["STREET"].fillna("")
    )
    out["TOWN_STREET"] = out["TOWN"].fillna("") + "|" + out["STREET"].fillna("")
    out["FLAT_TYPE_ORDER"] = out["FLAT_TYPE"].map(FLAT_TYPE_ORDER).astype("float32")

    out["FLAT_MODEL_GROUPED"] = out["FLAT_MODEL"].where(
        ~out["FLAT_MODEL"].isin(rare_models), "__rare__"
    )
    # Unseen future labels get the same treatment as rare training labels.
    known_models = set(train_model_counts.index)
    out.loc[~out["FLAT_MODEL"].isin(known_models), "FLAT_MODEL_GROUPED"] = "__rare__"

    lease_age = out["RENT_YEAR"] - pd.to_numeric(out["LEASE_COMMENCE_DATE"], errors="coerce")
    invalid_age = lease_age.lt(0) | lease_age.gt(100)
    out["LEASE_AGE_INVALID"] = invalid_age.astype("int8")
    out["LEASE_AGE_AT_RENT"] = lease_age.mask(invalid_age).astype("float32")
    out["LOG_FLOOR_AREA_SQM"] = np.log1p(pd.to_numeric(out["FLOOR_AREA_SQM"], errors="coerce"))

    out = out.drop(columns=["RENT_APPROVAL_DATE", "FURNISHED", "FEE"])
    return out


# Fit rare-model definition on training data only.
normalized_train_models = train_raw["FLAT_MODEL"].map(normalize_text)
train_model_counts = normalized_train_models.value_counts(dropna=False)
rare_models = set(train_model_counts[train_model_counts < RARE_MODEL_MIN_COUNT].index.dropna())

train = prepare_core(train_raw, rare_models)
test = prepare_core(test_raw, rare_models)

display(pd.DataFrame({
    "train_unique": train[["TOWN", "STREET", "FLAT_TYPE", "FLAT_MODEL_GROUPED"]].nunique(),
    "test_unique": test[["TOWN", "STREET", "FLAT_TYPE", "FLAT_MODEL_GROUPED"]].nunique(),
}))
print(f"Rare training flat models grouped: {sorted(rare_models)}")
print(f"Invalid lease ages — train: {train['LEASE_AGE_INVALID'].sum()}, test: {test['LEASE_AGE_INVALID'].sum()}")

## 4. HDB block and geospatial features

The HDB block table joins on normalized `TOWN + BLOCK + STREET`. Distances are calculated only once for each HDB block with the haversine metric, then merged into transactions. Only currently open MRT stations are used; primary schools include both primary-only and mixed `P1` schools.

In [ ]:
hdb = pd.read_csv(AUX_DIR / "sg-hdb-block.csv", dtype={"POSTAL_CODE": "string"})
for column in ["TOWN", "STREET"]:
    hdb[column] = hdb[column].map(normalize_text).astype("string")
hdb["BLOCK"] = hdb["BLOCK"].map(normalize_block).astype("string")

join_keys = ["TOWN", "BLOCK", "STREET"]
assert not hdb.duplicated(join_keys).any(), "HDB block join keys must be unique"

hdb["MAX_FLOOR"] = pd.to_numeric(hdb["MAX_FLOOR"], errors="coerce").replace(-1, np.nan)
hdb["YEAR_COMPLETED"] = pd.to_numeric(hdb["YEAR_COMPLETED"], errors="coerce").replace(-1, np.nan)
hdb["POSTAL_CODE"] = hdb["POSTAL_CODE"].replace({"NIL": pd.NA, "nil": pd.NA})
hdb["POSTAL_CODE"] = hdb["POSTAL_CODE"].str.zfill(6)
hdb["POSTAL_SECTOR"] = hdb["POSTAL_CODE"].str[:2]

hdb_features = hdb.rename(columns={
    "POSTAL_CODE": "HDB_POSTAL_CODE",
    "POSTAL_SECTOR": "HDB_POSTAL_SECTOR",
    "LATITUDE": "HDB_LATITUDE",
    "LONGITUDE": "HDB_LONGITUDE",
    "MAX_FLOOR": "HDB_MAX_FLOOR",
    "YEAR_COMPLETED": "HDB_YEAR_COMPLETED",
    "SUBZONE": "HDB_SUBZONE",
    "PLANNING_AREA": "HDB_PLANNING_AREA",
    "REGION": "HDB_REGION",
})


def nearest_distance_and_count(
    origins: pd.DataFrame,
    destinations: pd.DataFrame,
    radius_km: float = 1.0,
) -> tuple[np.ndarray, np.ndarray]:
    origin_rad = np.radians(origins[["HDB_LATITUDE", "HDB_LONGITUDE"]].to_numpy(float))
    destination_rad = np.radians(destinations[["LATITUDE", "LONGITUDE"]].to_numpy(float))
    tree = BallTree(destination_rad, metric="haversine")
    nearest_rad, _ = tree.query(origin_rad, k=1)
    counts = tree.query_radius(origin_rad, r=radius_km / EARTH_RADIUS_KM, count_only=True)
    return nearest_rad[:, 0] * EARTH_RADIUS_KM, counts.astype("int16")


mrt = pd.read_csv(AUX_DIR / "sg-mrt-stations.csv")
mrt = mrt.loc[mrt["STATUS"].map(normalize_text).eq("open")].copy()
# Multiple line codes can describe the same physical interchange.
mrt["STATION_KEY"] = (
    mrt["NAME"].map(normalize_text).astype("string") + "|" +
    mrt["LATITUDE"].round(5).astype(str) + "|" + mrt["LONGITUDE"].round(5).astype(str)
)
mrt = mrt.drop_duplicates("STATION_KEY")

malls = pd.read_csv(AUX_DIR / "sg-shopping-malls.csv").drop_duplicates(["LATITUDE", "LONGITUDE"])
schools = pd.read_csv(AUX_DIR / "sg-schools.csv")
primary_mask = schools["MAINLEVEL_CODE"].astype("string").str.lower().str.contains(
    r"primary|p1", regex=True, na=False
)
primary_schools = schools.loc[primary_mask].drop_duplicates(["LATITUDE", "LONGITUDE"])

geo = hdb_features[join_keys + ["HDB_LATITUDE", "HDB_LONGITUDE"]].copy()
for name, destinations in {
    "MRT": mrt,
    "MALL": malls,
    "PRIMARY_SCHOOL": primary_schools,
}.items():
    distance, count = nearest_distance_and_count(geo, destinations, radius_km=1.0)
    geo[f"DIST_TO_NEAREST_{name}_KM"] = distance.astype("float32")
    geo[f"{name}_COUNT_WITHIN_1KM"] = count

hdb_features = hdb_features.merge(geo.drop(columns=["HDB_LATITUDE", "HDB_LONGITUDE"]), on=join_keys, validate="one_to_one")


def merge_hdb_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.merge(hdb_features, on=join_keys, how="left", validate="many_to_one", indicator="_HDB_MERGE")
    out["HDB_MATCHED"] = out["_HDB_MERGE"].eq("both").astype("int8")
    out = out.drop(columns="_HDB_MERGE")
    building_age = out["RENT_YEAR"] - out["HDB_YEAR_COMPLETED"]
    invalid = building_age.lt(0) | building_age.gt(100)
    out["BUILDING_AGE_INVALID"] = invalid.fillna(False).astype("int8")
    out["BUILDING_AGE_AT_RENT"] = building_age.mask(invalid).astype("float32")
    return out


train = merge_hdb_features(train)
test = merge_hdb_features(test)

hdb_join_rates = {
    "train": float(train["HDB_MATCHED"].mean()),
    "test": float(test["HDB_MATCHED"].mean()),
}
display(pd.Series(hdb_join_rates, name="HDB join rate"))
assert min(hdb_join_rates.values()) >= 0.99, "Unexpectedly low HDB join coverage"

distance_columns = [column for column in train.columns if column.startswith("DIST_TO_NEAREST_")]
display(train[distance_columns].describe().T[["mean", "50%", "max"]])

## 5. Lagged COE market features

COE auction results are aggregated within each month and category, then shifted by one calendar month. This avoids using an auction result that might not have been known when a rental was approved. Category C is retained even though it is less directly related to HDB demand; its value can be judged during feature ablation.

In [ ]:
coe = pd.read_csv(AUX_DIR / "sg-coe-prices.csv")
coe["CATEGORY"] = coe["CATEGORY"].map(normalize_text).str.upper()
coe["COE_DATE"] = pd.to_datetime(
    coe["YEAR"].astype(str) + "-" + coe["MONTH"].astype(str),
    format="%Y-%b",
    errors="raise",
).dt.to_period("M").dt.to_timestamp()

for column in ["QUOTA_PREMIUM", "PREVAILING_QUOTA_PREMIUM", "QUOTA", "BIDS_RECEIVED"]:
    coe[column] = clean_number(coe[column])
coe["BID_TO_QUOTA_RATIO"] = coe["BIDS_RECEIVED"] / coe["QUOTA"].replace(0, np.nan)

coe_monthly_long = coe.groupby(["COE_DATE", "CATEGORY"], as_index=False).agg(
    PREMIUM_MEAN=("QUOTA_PREMIUM", "mean"),
    PREVAILING_PREMIUM_MEAN=("PREVAILING_QUOTA_PREMIUM", "mean"),
    QUOTA_SUM=("QUOTA", "sum"),
    BIDS_SUM=("BIDS_RECEIVED", "sum"),
    BID_TO_QUOTA_RATIO=("BID_TO_QUOTA_RATIO", "mean"),
)

coe_monthly = coe_monthly_long.pivot(index="COE_DATE", columns="CATEGORY")
coe_monthly.columns = [f"COE_{category}_{metric}" for metric, category in coe_monthly.columns]
coe_monthly = coe_monthly.sort_index()
full_months = pd.date_range(coe_monthly.index.min(), coe_monthly.index.max(), freq="MS")
coe_monthly = coe_monthly.reindex(full_months).shift(1)
coe_monthly.columns = [f"{column}_LAG1" for column in coe_monthly.columns]
coe_monthly.index.name = "RENT_DATE"
coe_monthly = coe_monthly.reset_index()

display(coe_monthly.tail(3))

## 6. Lagged stock-market features

Raw stock prices are not averaged across companies because price levels are not comparable. Instead, adjusted-close returns are calculated per symbol and summarized cross-sectionally each month. Extreme daily returns above 50% in absolute value are treated as data-quality anomalies for the volatility calculation. All monthly summaries are shifted by one month.

In [ ]:
stocks = pd.read_csv(AUX_DIR / "sg-stock-prices.csv")
stocks["DATE"] = pd.to_datetime(stocks["DATE"], errors="coerce")
stocks["SYMBOL"] = stocks["SYMBOL"].astype("string")
for column in ["OPEN", "HIGH", "LOW", "CLOSE", "ADJUSTED_CLOSE", "VOLUME"]:
    stocks[column] = pd.to_numeric(stocks[column], errors="coerce")
stocks = stocks.dropna(subset=["DATE", "SYMBOL", "ADJUSTED_CLOSE"]).sort_values(["SYMBOL", "DATE"])
stocks = stocks.loc[stocks["ADJUSTED_CLOSE"].gt(0)].copy()
stocks["STOCK_MONTH"] = stocks["DATE"].dt.to_period("M").dt.to_timestamp()
stocks["DAILY_RETURN"] = stocks.groupby("SYMBOL", observed=True)["ADJUSTED_CLOSE"].pct_change(fill_method=None)
stocks["DAILY_RETURN_CLEAN"] = stocks["DAILY_RETURN"].mask(stocks["DAILY_RETURN"].abs().gt(0.50))

daily_vol = stocks.groupby(["SYMBOL", "STOCK_MONTH"], observed=True)["DAILY_RETURN_CLEAN"].std().rename("DAILY_VOL")
month_end = stocks.groupby(["SYMBOL", "STOCK_MONTH"], observed=True).tail(1).copy()
month_end["MONTHLY_RETURN"] = month_end.groupby("SYMBOL", observed=True)["ADJUSTED_CLOSE"].pct_change(fill_method=None)
month_end = month_end.merge(daily_vol.reset_index(), on=["SYMBOL", "STOCK_MONTH"], how="left", validate="one_to_one")

# Cross-sectional winsorization is performed independently within each month.
def winsorize_month(series: pd.Series) -> pd.Series:
    if series.notna().sum() < 5:
        return series
    lower, upper = series.quantile([0.01, 0.99])
    return series.clip(lower, upper)


month_end["MONTHLY_RETURN_WINSORIZED"] = month_end.groupby("STOCK_MONTH", observed=True)["MONTHLY_RETURN"].transform(winsorize_month)
market_monthly = month_end.groupby("STOCK_MONTH", as_index=False, observed=True).agg(
    STOCK_RETURN_MEDIAN=("MONTHLY_RETURN", "median"),
    STOCK_RETURN_MEAN_WINSORIZED=("MONTHLY_RETURN_WINSORIZED", "mean"),
    STOCK_RETURN_STD=("MONTHLY_RETURN", "std"),
    STOCK_POSITIVE_SHARE=("MONTHLY_RETURN", lambda values: values.gt(0).mean()),
    STOCK_DAILY_VOL_MEDIAN=("DAILY_VOL", "median"),
    STOCK_ACTIVE_SYMBOL_COUNT=("SYMBOL", "nunique"),
    STOCK_RETURN_SYMBOL_COUNT=("MONTHLY_RETURN", "count"),
)
market_monthly = market_monthly.set_index("STOCK_MONTH").sort_index()
full_months = pd.date_range(market_monthly.index.min(), market_monthly.index.max(), freq="MS")
market_monthly = market_monthly.reindex(full_months).shift(1)
market_monthly.columns = [f"{column}_LAG1" for column in market_monthly.columns]
market_monthly.index.name = "RENT_DATE"
market_monthly = market_monthly.reset_index()

display(market_monthly.tail(3))

## 7. Merge monthly features and finalize schemas

Missing lagged values at the beginning of the data are expected: January 2021 has no December 2020 auxiliary record. They remain numeric missing values for the downstream model to handle. Categorical missing values receive an explicit token.

In [ ]:
def merge_monthly_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.merge(coe_monthly, on="RENT_DATE", how="left", validate="many_to_one")
    out = out.merge(market_monthly, on="RENT_DATE", how="left", validate="many_to_one")
    return out


train = merge_monthly_features(train)
test = merge_monthly_features(test)

categorical_features = [
    "TOWN", "BLOCK", "STREET", "FLAT_TYPE", "FLAT_MODEL", "FLAT_MODEL_GROUPED",
    "PROPERTY_KEY", "TOWN_STREET", "HDB_POSTAL_CODE", "HDB_POSTAL_SECTOR",
    "HDB_SUBZONE", "HDB_PLANNING_AREA", "HDB_REGION",
]
for column in categorical_features:
    train[column] = train[column].astype("string").fillna("__missing__")
    test[column] = test[column].astype("string").fillna("__missing__")

# Restore deterministic order and place target last.
train = train.sort_values("ROW_ID").reset_index(drop=True)
test = test.sort_values("ROW_ID").reset_index(drop=True)
feature_columns = [column for column in test.columns]
train = train[feature_columns + [target]]

non_feature_columns = ["ROW_ID", "RENT_DATE"]
model_features = [column for column in feature_columns if column not in non_feature_columns]

assert train["ROW_ID"].tolist() == list(range(len(train_raw)))
assert test["ROW_ID"].tolist() == list(range(len(test_raw)))
assert train.drop(columns=[target]).columns.tolist() == test.columns.tolist()
assert "RENT_PER_SQM" not in train.columns
train_numeric = train.select_dtypes(include="number").to_numpy(dtype=float, na_value=np.nan)
test_numeric = test.select_dtypes(include="number").to_numpy(dtype=float, na_value=np.nan)
assert not np.isinf(train_numeric).any()
assert not np.isinf(test_numeric).any()

summary = pd.DataFrame({
    "rows": [len(train), len(test)],
    "columns": [train.shape[1], test.shape[1]],
    "model_features": [len(model_features), len(model_features)],
    "numeric_missing_cells": [
        int(train.select_dtypes(include=np.number).isna().sum().sum()),
        int(test.select_dtypes(include=np.number).isna().sum().sum()),
    ],
}, index=["train_preprocessed", "test_preprocessed"])
display(summary)

missing_report = (
    train.isna().mean().sort_values(ascending=False).rename("train_missing_rate").to_frame()
    .join(test.isna().mean().rename("test_missing_rate"), how="outer")
)
display(missing_report.loc[missing_report.max(axis=1).gt(0)].head(20))

## 8. Save processed artifacts

The metadata file records the feature contract and key preprocessing decisions. `ROW_ID` must be retained until submission creation, where predictions are sorted by it before being placed in the sample-submission layout.

In [ ]:
metadata = {
    "target": target,
    "row_id": "ROW_ID",
    "date_column": "RENT_DATE",
    "non_feature_columns": non_feature_columns,
    "model_features": model_features,
    "categorical_features": [column for column in categorical_features if column in model_features],
    "rare_model_min_count": RARE_MODEL_MIN_COUNT,
    "rare_models_from_train": sorted(rare_models),
    "hdb_join_rates": hdb_join_rates,
    "train_rows": len(train),
    "test_rows": len(test),
    "preprocessing_decisions": {
        "duplicates_removed": False,
        "target_outliers_removed": False,
        "constant_columns_dropped": ["FURNISHED", "FEE"],
        "invalid_ages": "set to missing and flagged",
        "coe_features": "monthly aggregation lagged by one month",
        "stock_features": "per-symbol returns, robust monthly aggregation, lagged by one month",
        "target_derived_features": "forbidden",
    },
}

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    train.to_csv(OUTPUT_DIR / "train_preprocessed.csv", index=False, date_format="%Y-%m-%d")
    test.to_csv(OUTPUT_DIR / "test_preprocessed.csv", index=False, date_format="%Y-%m-%d")
    with (OUTPUT_DIR / "preprocessing_metadata.json").open("w", encoding="utf-8") as handle:
        json.dump(metadata, handle, ensure_ascii=False, indent=2)
    print(f"Saved processed files to: {OUTPUT_DIR}")
    for path in sorted(OUTPUT_DIR.glob("*")):
        print(f"- {path.name}: {path.stat().st_size / 1_048_576:.2f} MiB")

## 9. Modeling handoff

Recommended next step: start with a CatBoost regressor and blocked chronological validation.

1. Sort training rows by `RENT_DATE` and use expanding or rolling time splits; do not use a random split as the primary score.
2. Exclude `ROW_ID` and `RENT_DATE` from the model matrix. Keep `MONTH_INDEX` and the calendar features.
3. Pass the categorical column names from `preprocessing_metadata.json` directly to CatBoost. No one-hot encoding or scaling is required for this baseline.
4. Compare core-only, core + HDB/geospatial, and core + all lagged market features through ablation.
5. Fit preprocessing decisions using training folds only if later adding target encoding, imputation learned from data, or supervised feature selection.
6. For Kaggle submission, predict test rows in `ROW_ID` order and use exactly the column structure from `data/example-submission.csv`.

The preprocessing intentionally does not use the target for filtering, encoding, imputation, or feature construction.